# Program performance

**Is remediation effort landing on the findings that matter, and can we close faster than
risk arrives?**

The Cisco Kenna / Cyentia *Prioritization to Prediction* family. **Coverage** is
`TP / (TP + FN)` — of all high-risk vulnerabilities, what share did we remediate.
**Efficiency** is `TP / (TP + FP)` — of everything we remediated, what share was actually
high-risk. They pull against each other, so they are always shown together; P2P vol. 2's
industry baseline is 70% coverage at 18.5% efficiency.

## How to read this notebook

Every cell answers one question and shows one thing. Run them in order the first time; after
that any cell can be re-run on its own.

**Set the widgets at the top before you run anything.** `catalog` has no default on purpose.
Set the notebook to **Run accessed commands** (the dropdown beside *Run all*) if you want a
widget change to re-run the cells that depend on it — otherwise you will change the filter and
read a chart drawn under the old one.

Everything here reads one scan, pinned in cell 1. Charts that span scans say so in their title.

In [ ]:
import os, sys

_paths = []
try:
    _paths.append(dbutils.widgets.get("module_path"))
except Exception:  # noqa: BLE001 -- the widget does not exist yet on a first run
    pass
_here = os.getcwd()
_paths += [_here, os.path.dirname(_here)]
for _p in _paths:
    if _p and os.path.exists(os.path.join(_p, "panels.py")):
        sys.path.insert(0, _p)
        break
else:
    raise RuntimeError("brick modules are not on sys.path -- see brick/README.md, step 2")

import panels, figures, tiles

PAGE = {"quadrant": ("fn", ["tp", "fp", "fn", "tn", "unknown_open", "unknown_remediated"])}

panels.declare_widgets(**PAGE)
ctx = panels.context(spark, **{name: str(spec[0]) for name, spec in PAGE.items()})
displayHTML(tiles.scan_zone_from(panels.last_scan(spark, ctx).first()))

## Coverage and efficiency, with the width of their doubt

In [ ]:
displayHTML(tiles.program_hero(panels.program_headline(spark, ctx).first()))

## What the effort landed on

Deliberately uncoloured. A red-washed *high risk, still open* cell would make the other
three unreadable by comparison, and shouting is not the same as informing.

The bottom row is not a fifth quadrant. A finding whose exploit signal was never captured
is **not** evidence of safety — it leaves both rates, and the published ranges beside each
figure above are what that population could do to them.

In [ ]:
displayHTML(tiles.matrix(panels.program_headline(spark, ctx).first().asDict()))

Pick a cell with the `quadrant` widget and the lifecycles behind it come back in the
result grid below — which sorts, filters and exports to CSV, so there is no drill-down
UI here worth writing.

In [ ]:
display(panels.quadrant(spark, ctx, ctx.param('quadrant')))

## Over time

Both rates in percent, plus the prevalence baseline — the share you would hit by picking
findings at random. **This third series is an addition, not a port**: the GAS chart draws
two. It is here because efficiency is meaningless without knowing what chance would have
achieved, but it is a property of the register's population, not a target to beat.

A NULL rate is a break in the line. It means the denominator was empty, which is not 0%.

In [ ]:
figures.render(
    figures.describe(
        figures.trend(
            panels.trend(
                spark, ctx, ["coverage_pct", "efficiency_pct", "prevalence_pct"],
                family="program",
            ).toPandas(),
            "scan_ts",
            [
                figures.Series("coverage_pct", "Coverage", "#2563eb", symbol="circle"),
                figures.Series("efficiency_pct", "Efficiency", "#0d9488", dash="6,4",
                               symbol="square"),
                figures.Series("prevalence_pct", "Random baseline", "#94a3b8",
                               dash="2,3", symbol="diamond"),
            ],
            y_range=[0, 100],
            y_unit="percent",
        ),
        "Coverage and efficiency per scan, against the prevalence baseline. Gaps are scans where the population was empty, not zero rates.",
    )
)

## How high risk is decided

In [ ]:
displayHTML(
    tiles.rule_card(
        panels.rule_sentence(ctx),
        [
            {"text": "Listed in the CISA KEV catalog",
             "fired": _clauses["kev"], "missing": _clauses["kev_missing"]},
            {"text": "A public exploit exists",
             "fired": _clauses["exploit"], "missing": _clauses["exploit_missing"]},
            {"text": f"EPSS at or above {ctx.rule.epss_threshold:.2f}",
             "fired": _clauses["epss"], "missing": _clauses["epss_missing"]},
        ],
        signal_coverage=panels.program_headline(spark, ctx)
            .first()["signal_coverage_pct"],
    )
) if (_clauses := panels.signal_clauses(spark, ctx).first().asDict()) else None

Every point below is one of the seven possible rules; the filled diamond is the one in
force, carrying the published uncertainty on both axes. Up and to the right is better on
both counts, which almost never happens — that trade-off is the whole point of the chart.

Labels sit on the points rather than in a legend: the question here is *which point is
which rule*, and a legend makes that a lookup.

In [ ]:
figures.render(
    figures.describe(
        figures.scatter_bounds(
            panels.rule_sweep(spark, ctx).toPandas(),
            lo_x="coverage_lo", hi_x="coverage_hi",
            lo_y="efficiency_lo", hi_y="efficiency_hi",
            baseline=panels.program_headline(spark, ctx)
                .first()["prevalence_pct"],
            baseline_label="random baseline",
        ),
        "Coverage against efficiency for each candidate high-risk rule. The filled diamond is the rule in force; its error bars are the published range the unclassified population could move it through.",
    )
)

## Remediation capacity

Monthly close rate against the *about one in ten open findings per month* benchmark.

Two columns are worth reading carefully. `tag` marks months that predate the first scan —
their opens and closes are back-dated from the API's own dates rather than watched, so
they are not evidence of capacity. `closed_observed` is reconciliation's own count of
closures, an independent route to the same number; where the two disagree, one of them is
wrong, and publishing both is what lets you notice.

In [ ]:
%sql
SELECT month, open_at_start, opened, closed, closed_observed,
       mmcr, net, net_pct, verdict,
       CASE WHEN reconstructed THEN 'reconstructed'
            WHEN partial       THEN 'in progress'
            ELSE '' END AS tag
FROM   v_capacity
ORDER  BY month DESC
LIMIT  12

The same over the **high-risk population only** — GAS's *high-risk net*. This one is
recomputed in the notebook: `metrics.capacity_by_month` has taken the flag since v2 but
the pipeline has never passed it, so the published table only holds the all-findings
version.

In [ ]:
display(panels.capacity(spark, ctx, months=12, high_risk_only=True))

In [ ]:
displayHTML(
    tiles.methodology(
        [
            {"term": "Coverage",
             "definition": "TP / (TP + FN). Of all high-risk lifecycles, the share remediated."},
            {"term": "Efficiency",
             "definition": "TP / (TP + FP). Of everything remediated, the share that was high risk."},
            {"term": "The published range",
             "definition": "The two extremes of re-labelling every unclassified lifecycle. Its width is the size of the doubt, not a confidence interval."},
            {"term": "Unclassified",
             "definition": "A lifecycle with an exploit signal that was never captured. Never counted as low risk -- that single mistake inflates efficiency and deflates coverage at once."},
            {"term": "Sticky signals",
             "definition": "KEV and exploit flags go null -> false -> true and never back; EPSS keeps its peak. A published point never moves for reasons unrelated to remediation."},
            {"term": "MMCR",
             "definition": "Closed in a month over open at its start. The benchmark is about one in ten."},
            {"term": "Reconstructed months",
             "definition": "Months before the first scan, back-dated from the API's own dates. Excluded from the headline mean."},
            {"term": "What counts as remediated",
             "definition": "A lifecycle with a resolution date -- learned from the API, or inferred from the finding disappearing. The split is published per severity in 01_mttr_sla."},
        ],
        summary="How these numbers are calculated",
    )
)